# Exploratory Data Analysis

In this notebook we will look into the speedtest data gathered from ookla and draw some useful insights from the dataset. Additional dataset would also be utilized to make some meaningful insights into the appropriately combined wholistic data

## File Exploration

In the following block we see all the files available for this kernel. The relevant libraries are also imported to process the data extracted from the dataset.

In [1]:
%load_ext cudf.pandas

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import regex as re # For String searches
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2020_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2020_quarter_01.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_01.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_03.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2020_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2020_quarter_03.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2022_quarter_01.csv
/home/dias-benchmarks/notebooks/gk

In [3]:
data = pd.read_csv('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2021_quarter_03.csv')
data.head()

,Name,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency
0,Afghanistan,897,"2,312","9,003","3,983","4,823",70,"25,067,407",217,231,33
1,Åland Islands,284,580,"1,341","58,693","82,767",11,0,27,47,215
2,Albania,"7,215","38,905","104,752","17,354","25,803",20,"3,153,731",100,127,159
3,Algeria,"16,056","70,929","413,666","1,508","9,057",43,"32,854,159",234,211,68
4,American Samoa,80,202,900,"14,345","33,078",18,"64,051",119,104,171


In [4]:
data.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               236 non-null    object
 1   Number of Records  236 non-null    object
 2   Devices            236 non-null    object
 3   Tests              236 non-null    object
 4   Avg. Avg U Kbps    236 non-null    object
 5   Avg. Avg D Kbps    236 non-null    object
 6   Avg Lat Ms         236 non-null    int64
 7   Avg. Pop2005       236 non-null    object
 8   Rank Upload        236 non-null    int64
 9   Rank Download      236 non-null    int64
 10  Rank Latency       236 non-null    int64
dtypes: int64(4), object(7)
memory usage: 24.3+ KB


In [5]:
Mobile_df = pd.DataFrame([],columns=data.columns)
Broadband_df = pd.DataFrame([],columns=data.columns)

def col_name_corrections(df,correction_pair):
    if set(df.columns).intersection(set(correction_pair.keys())):
        df.rename(columns=correction_pair,inplace=True)
    return df

for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        meta_info = filename.split('/')[-1]
        data = pd.read_csv(dirname+'/'+filename,thousands=r',').convert_dtypes()
        data = col_name_corrections(data,{'Number of Record':'Number of Records'})
        data['Year'] = np.int64(re.search('year_(.*)_quarter',meta_info).group(1))
        data['Quarter'] = np.int64(re.search('quarter_(.*).csv',meta_info).group(1))
        if 'mobile' in meta_info:
            Mobile_df = pd.concat([Mobile_df,data])
        else:
            Broadband_df = pd.concat([Broadband_df,data]) 
print(Broadband_df.shape)
print(Mobile_df.shape)
Mobile_df = Mobile_df.astype({'Year':np.int64,'Quarter':np.int64})
Broadband_df = Broadband_df.astype({'Year':np.int64,'Quarter':np.int64})
Mobile_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)
Broadband_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)

(2597, 13)
(2487, 13)


It can be seen that there are more broadband rows than Mobile rows. This is a point that should be noted each row corresponds to a country's statistics in the particular year and quarter. Missing data indicates lack of speed test data from the country.

In [6]:
Mobile_df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 2487 entries, 0 to 232
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               2487 non-null   object
 1   Number of Records  2487 non-null   int64
 2   Devices            2487 non-null   int64
 3   Tests              2487 non-null   int64
 4   Avg. Avg U Kbps    2487 non-null   int64
 5   Avg. Avg D Kbps    2487 non-null   int64
 6   Avg Lat Ms         2487 non-null   int64
 7   Avg. Pop2005       2487 non-null   int64
 8   Rank Upload        2487 non-null   int64
 9   Rank Download      2487 non-null   int64
 10  Rank Latency       2487 non-null   int64
 11  Year               2487 non-null   int64
 12  Quarter            2487 non-null   int64
dtypes: int64(12), object(1)
memory usage: 286.8+ KB


# -- STEFANOS -- Replicate Data

In [8]:

# factor = 20
Mobile_df = pd.concat([Mobile_df]*factor, ignore_index=True)
Mobile_df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 49740 entries, 0 to 49739
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               49740 non-null  object
 1   Number of Records  49740 non-null  int64
 2   Devices            49740 non-null  int64
 3   Tests              49740 non-null  int64
 4   Avg. Avg U Kbps    49740 non-null  int64
 5   Avg. Avg D Kbps    49740 non-null  int64
 6   Avg Lat Ms         49740 non-null  int64
 7   Avg. Pop2005       49740 non-null  int64
 8   Rank Upload        49740 non-null  int64
 9   Rank Download      49740 non-null  int64
 10  Rank Latency       49740 non-null  int64
 11  Year               49740 non-null  int64
 12  Quarter            49740 non-null  int64
dtypes: int64(12), object(1)
memory usage: 5.2+ MB


In [9]:

Broadband_df = pd.concat([Broadband_df]*factor, ignore_index=True)
Broadband_df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 51940 entries, 0 to 51939
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               51940 non-null  object
 1   Number of Records  51940 non-null  int64
 2   Devices            51940 non-null  int64
 3   Tests              51940 non-null  int64
 4   Avg. Avg U Kbps    51940 non-null  int64
 5   Avg. Avg D Kbps    51940 non-null  int64
 6   Avg Lat Ms         51940 non-null  int64
 7   Avg. Pop2005       51940 non-null  int64
 8   Rank Upload        51940 non-null  int64
 9   Rank Download      51940 non-null  int64
 10  Rank Latency       51940 non-null  int64
 11  Year               51940 non-null  int64
 12  Quarter            51940 non-null  int64
dtypes: int64(12), object(1)
memory usage: 5.5+ MB


In [10]:
Mobile_df.head()

,Name,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
0,Afghanistan,789,1241,3541,3099,6387,70,25067407,220,220,32,2020,1
1,Albania,1626,4513,7228,11414,40238,26,3153731,75,42,197,2020,1
2,Algeria,14174,48699,124160,6150,7724,65,32854159,197,213,35,2020,1
3,American Samoa,23,33,63,8488,22792,62,64051,156,111,38,2020,1
4,Andorra,45,83,110,18160,72764,39,73483,9,6,112,2020,1


In [11]:
%%time
unique_countries_broadband = Broadband_df.groupby('Name').count()
unique_countries_broadband.head()

CPU times: user 49.8 ms, sys: 16.4 ms, total: 66.2 ms
Wall time: 88.3 ms


,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
Name,,,,,,,,,,,,
Afghanistan,220,220,220,220,220,220,220,220,220,220,220,220
Albania,220,220,220,220,220,220,220,220,220,220,220,220
Algeria,220,220,220,220,220,220,220,220,220,220,220,220
American Samoa,220,220,220,220,220,220,220,220,220,220,220,220
Andorra,220,220,220,220,220,220,220,220,220,220,220,220
